In [1]:
import sys
sys.path.append(r"C:\Users\user\Desktop\nba_copilot")

In [2]:
# test rapide
from Composant4.cba_service import dispatch

# État d'une équipe
print(dispatch("get_team_state", {"team_id": 1, "saison": "2025-26"}))

# Vérifier un move
print(dispatch("check_move", {
    "team_id": 1,
    "saison": "2025-26",
    "total_cap_hit": 145_000_000,
    "total_base_salary": 140_000_000,
    "nb_joueurs": 14,
    "move_type": "SIGN",
    "salary_in": 14_104_000,
    "exception": "MLE_NON_TAXPAYER",
    "description": "Signer via MLE non-taxpayer",
}))

# Proposer les meilleurs moves
print(dispatch("propose_moves", {"team_id": 1, "saison": "2025-26", "top_n": 5}))

tool_get_team_state erreur: Traceback (most recent call last):
  File "C:\Users\user\Desktop\nba_copilot\Composant4\cba_service.py", line 508, in tool_get_team_state
    state = _load_team_state(inp.team_id, inp.saison)
  File "C:\Users\user\Desktop\nba_copilot\Composant4\engine.py", line 164, in _load_team_state
    return TeamPayroll(
        team_id           = team_id,
    ...<6 lines>...
        bi_annual_used    = False,
    )
  File "C:\Users\user\AppData\Roaming\Python\Python313\site-packages\pydantic\main.py", line 250, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
pydantic_core._pydantic_core.ValidationError: 1 validation error for TeamPayroll
team_name
  Field required [type=missing, input_value={'team_id': 1, 'saison': ...'bi_annual_used': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

tool_propose_moves ValueError: 1 validation error for TeamPayroll
team_name
  

❌ Erreur : Erreur interne : 1 validation error for TeamPayroll
team_name
  Field required [type=missing, input_value={'team_id': 1, 'saison': ...'bi_annual_used': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
❌ Erreur : Input invalide : 2 validation errors for CheckMoveInput
move_type
  Value error, move_type invalide: 'SIGN'. Valeurs valides: {'waive', 'trade', 'sign', 'extend'} [type=value_error, input_value='SIGN', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
exception
  Value error, exception invalide: 'MLE_NON_TAXPAYER'. Valeurs valides: {'mini_mle', 'bi_annual', 'none', 'mle_taxpayer', 'mle_non_taxpayer'} [type=value_error, input_value='MLE_NON_TAXPAYER', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
❌ Erreur : 1 validation error for TeamPayroll
team_name
  Field required [type=missing, input_value={'team_id': 1, 'saison': 

In [10]:
import pandas as pd
from sqlalchemy import create_engine, text
from Composant3.config import DB_URL

engine = create_engine(DB_URL)

with engine.connect() as conn:
    tables = pd.read_sql("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema='public'
    """, conn)
    print("Tables disponibles :")
    print(tables)

Tables disponibles :
           table_name
0             players
1       player_id_map
2        player_stats
3           contracts
4               teams
5       team_payrolls
6          risk_flags
7   watchlist_entries
8              alerts
9     player_features
10     training_pairs


In [11]:
from Composant4.engine import _get_conn
import pandas as pd

with _get_conn() as conn:
    print("=== ÉQUIPES (teams) ===")
    teams_df = pd.read_sql("SELECT * FROM teams LIMIT 5", conn)
    print(teams_df)
    
    print("\n=== PAYROLLS (team_payrolls) ===")
    payrolls_df = pd.read_sql("SELECT * FROM team_payrolls LIMIT 5", conn)
    print(payrolls_df)

=== ÉQUIPES (teams) ===
Empty DataFrame
Columns: [team_id, team_name, abbreviation, conference, division]
Index: []

=== PAYROLLS (team_payrolls) ===
Empty DataFrame
Columns: [team_id, saison, total_cap_hit, total_base_salary, nb_joueurs]
Index: []


C:\Users\user\AppData\Local\Temp\ipykernel_33780\2007841948.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  teams_df = pd.read_sql("SELECT * FROM teams LIMIT 5", conn)
C:\Users\user\AppData\Local\Temp\ipykernel_33780\2007841948.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  payrolls_df = pd.read_sql("SELECT * FROM team_payrolls LIMIT 5", conn)


In [12]:
from Composant4.engine import _get_conn

with _get_conn() as conn:
    with conn.cursor() as cursor:
        # 1. Insérer une équipe fictive avec l'ID 1
        cursor.execute("""
            INSERT INTO teams (team_id, team_name, abbreviation, conference, division)
            VALUES (1, 'Boston Celtics', 'BOS', 'East', 'Atlantic')
            ON CONFLICT DO NOTHING;
        """)
        
        # 2. Insérer un payroll correspondant pour la saison 2025-26
        cursor.execute("""
            INSERT INTO team_payrolls (team_id, saison, total_cap_hit, total_base_salary, nb_joueurs)
            VALUES (1, '2025-26', 150000000, 140000000, 15)
            ON CONFLICT DO NOTHING;
        """)
    conn.commit()

print("Données de test insérées avec succès !")

Données de test insérées avec succès !
